In [ ]:
#Last Edited: 2025/06/20
#File changed more recently (2025/11/18) to commit (with Cell outputs deleted)

In [ ]:
#This is essentially a copy of "ADCP_Anomaly_Training.ipynb", but editted to use GPU resources on DRAC

In [ ]:
#I created this ipynb file, and many of the .py files using the help of chatGPT.  
#This was my initial prompt:

'''
So, I'd like to train an ML model.  I have a data set consisting of 4000 h5-format files.  Each file contains 24 hours of data.  There is a data group called '/data' with many multi channel variables.  There is also a set of annotations that indicate the classication of the data when anomalies are present.  Generally the class-imbalance for anomalous data is very large.  Probably <1 day with anomalous data per 300 days anomaly free. I want to do several things:

1) Do pre-training on the "good"/"anomaly free" data
2) Train on classified data.

Things I will need to do:
1) Have a data loader that loads the data
- the data is for an ADCP with 3 beams.  Each beam collects data for beam velocity, beam backscatter, and beam correlation.  Since the beams are kind of independant, it probably makes sense to organize the data such that there are 3 input channels [velocity, backscatter, correlation], and each beam can be fed through the model independantly.  
-The "anomalous" data may only occur for 6 hours within a 24 hour period, and I'd like the model to identify specifically which data is anomalous (not just binary yes/no for the entire 24 hour period).  I don't know if it makes sense to create a layer mask for the data, or if some kind of time-series label would be more appropriate (or even possible)

2) Define a model architecture / type
 - I want this to be set up so that I can experiment with differnet options for this, and have this be pretty modular.  Any suggestions would be appreciated 
 - I also want to have the option to be able to randomly tile the inputs, to add more variance for training, but this might not be needed

3) Have a loss function that is weighted to account for class-imbalance.  I've used graduated dice loss in the past but not sure what other options are available

'''

#It then gave me a bunch of info, but no code, and I responded with 

'''
so each input channel is 2D (time, range), and with multiple channels is 3D.  I guess I don't need to do anything fancy like making a mask for the annotations since it's not semantic segmentation, so the output could probably by 1D (time), with values like 0 (normal), 1, 2, 3, to indicate the class. Right now the annotations are start/end time and index, so I'll need to create a time-series label with the data loader.  

 I prefer pytorch.  I think a CNN is the minimum I'll want to use.  I'll also need a normalization step.  I like F1-score for an evaluation metric, and maybe a combo of cross entropy and dice loss, 

Can you give me code for all of this please?  Maybe start with the data loader. If there is a limit on tokens, do just the data loader, then ask me to say continue for each subsequent section of code
'''

#And from there it pretty much gave me all of this.  Now I have he task of testing and debugging the code and actually making it work for my data

In [ ]:

'''
To set up the environment:

conda create -n adcp_anomaly_env python=3.10
conda activate adcp_anomaly_env
pip install -r requirements.txt

OR

python -m venv adcp_anomaly_env
adcp_anomaly_env\Scripts\activate
pip install -r requirements.txt

'''

In [ ]:
#Some changes:
# I updated dataset_loader so that MINOR A and MINOR B classes are not included
# By this I mean, they are not excluded from training, they just have class = 0 (same as normal data)
# I chose to do this because it was super-ceding proper anomalous data, and really skewing the model training
# => This change lead to SIGNIFICANT improvement in classification

#Session Settings on JupyterHub
#Memory: 15000
#Cores: 4
#GPUs: 1

#Kernel: SSAMBA Kernel on Scratch



In [ ]:
# Cell 1: Imports & Setup

import os
import torch
from torch.utils.data import DataLoader, random_split

# Add repo root to Python path - Needed to import from src folder
import sys
from pathlib import Path
repo_root = Path().resolve().parent  # notebooks/ → ADCP-CNN-QAQC
sys.path.append(str(repo_root))

from dataset_loader_noEmbed import ADCPDataset  # your custom dataset
#from src.dataset_loader import ADCPDataset  # your custom dataset
from src.model import TemporalCNN # CNNClassifier  # your model
from src.utils import seed_everything, get_class_weights, combined_loss, train_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed_num = 42
seed_everything(seed_num)


#IMPORTANT NOTE: ALSO NEED TO ADJUST THIS IN "utils.py" TO MATCH, in "def train_model()"

USE_WANDB = True  # TODO: Set to True to enable logging - Keep false while in development/debugging
if USE_WANDB:
    import wandb
    
DEBUG_MODE = False

In [2]:
# Cell 2: Initialize wandb
#53842d9970aafed0ab407079e403fe03469dcb33

from types import SimpleNamespace

if USE_WANDB:
    wandb.init(project="adcp-anomaly-detection", #dir="/scratch/ML_ADCP/wandb_runs", 
            config={
                "model": "TemporalCNN",
                "epochs": 20,
                "batch_size": 16,
                "lr": 1e-3,
                "loss_alpha": 0.5,
                "optimizer": "Adam"
            })
    config = wandb.config
else:
    config = SimpleNamespace(**{
        "model": "TemporalCNN",
        "epochs": 20,
        "batch_size": 16, #2,
        "lr": 1e-3,
        "loss_alpha": 0.5,
        "optimizer": "Adam"})

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: slonimer (slonimer-university-of-victoria) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


In [15]:
#DEBUG
'''
#Get a list of files from the directory:
data_folder= r"F:\Documents\Projects\ML\ADCP_ML\h5_24h_files\\"
#annotation_file = "/path/to/annotations.json"
file_list = os.listdir(data_folder)
h5_files = {k for k in file_list if os.path.splitext(k)[1] == ".h5"}
h5_files = sorted(h5_files)  # Sorts alphabetically

#print(os.path.splitext(file_list[0]))
#print(h5_files)

h5_paths = []
for filename in h5_files:
    h5_paths.append(data_folder + filename) 

#print(len(h5_paths))

#WHILE DEBUGGING, ONLY US A FEW FILES
#CHANGE THIS LATER
file_idx = 3567
print(h5_paths[file_idx])             

'''

F:\Documents\Projects\ML\ADCP_ML\h5_24h_files\\20240406T000000_20240406T235959.h5


In [ ]:
#DEBUG
import importlib
import dataset_loader_noEmbed
importlib.reload(dataset_loader_noEmbed)
#import src.dataset_loader
#importlib.reload(src.dataset_loader)

from dataset_loader_noEmbed import ADCPDataset  # your custom dataset
# from src.dataset_loader import ADCPDataset

In [5]:
# Cell 3: Load Dataset

from sklearn.model_selection import train_test_split

#Get a list of files from the directory:
data_folder = "/scratch/slonimer/ML_ADCP/BACAX_24hr_h5/"
#data_folder= r"F:\Documents\Projects\ML\ADCP_ML\h5_24h_files\\"
file_list = os.listdir(data_folder)
h5_files = {k for k in file_list if os.path.splitext(k)[1] == ".h5"}
#Files are inherently NOT in order in python! So if you want them in order, need to do this:
h5_files = sorted(h5_files)  # Sorts alphabetically

#print(os.path.splitext(file_list[0]))
#print(h5_files)

h5_paths = []
for filename in h5_files:
    h5_paths.append(data_folder + filename) 

#print(len(h5_paths))

# Load anomaly filenames from a text file
with open("annotated_files.txt", "r") as f:
    anomaly_files = set(line.strip() for line in f if line.strip())
    
#Define anomaly paths before truncating h5_paths
anomaly_paths = [p for p in h5_paths if os.path.basename(p) in anomaly_files]

DEBUG_MODE = 0
if DEBUG_MODE:
    #WHILE DEBUGGING, ONLY USE A FEW FILES
    #NEED THIS FOR DEBUGGING - POTENTIALLY USES 13 GB OF MEMORY
    num_files = 200 # 1000 makes the kernel crash w 2400 MB memory
    
    h5_paths = h5_paths[:num_files]    

    #file_idx = 3567
    #h5_paths = h5_paths[file_idx-5 : file_idx+5 ]                                   

#full_dataset = ADCPDataset(h5_paths) # (data_dir)

normal_paths = [p for p in h5_paths if os.path.basename(p) not in anomaly_files]


# Example: 70% train, 20% val, 10% test
#total_size = len(full_dataset)
#train_size = int(0.7 * total_size)
#val_size = int(0.20 * total_size)
#test_size = total_size - train_size - val_size  # ensure all samples are used

#train_dataset, val_dataset, test_dataset = random_split(
#    full_dataset, [train_size, val_size, test_size]

#train_size = int(0.8 * len(full_dataset))
#val_size = len(full_dataset) - train_size
#train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Split anomaly files
an_train, an_temp = train_test_split(anomaly_paths, test_size=0.3, random_state=seed_num) #70/30 split
an_val, an_test = train_test_split(an_temp, test_size=0.33, random_state=seed_num) #20/10 split 

# Split normal files
n_train, n_temp = train_test_split(h5_paths, test_size=0.3, random_state=seed_num) #70/30 split
n_val, n_test = train_test_split(n_temp, test_size=0.33, random_state=seed_num) #20/10 split 

# Combine
train_files = an_train + n_train
val_files = an_val + n_val
test_files = an_test + n_test

#Create the datasets:
train_dataset = ADCPDataset(train_files)
val_dataset = ADCPDataset(val_files)
test_dataset = ADCPDataset(test_files)

print('train/val/test datasets grabbed')


train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, num_workers=4)


train/val/test datasets grabbed


In [24]:
#DEBUG
def _load_h5(path):
    with h5py.File(path, 'r') as f:
        data = f['/data']
        beam_data = []

        for beam in [0, 1, 2]:
            v = data['velocity'][beam]      # shape (time, range)
            b = data['backscatter'][beam]
            c = data['correlation'][beam]

            beam_stack = np.stack([v, b, c], axis=0)  # [3, time, range]
            beam_data.append(beam_stack)

        timestamps = f['/data/time'][:]
        return beam_data, timestamps
        
for path in h5_paths[0:5]:
    beam_data, timestamps = _load_h5(path)
    with h5py.File(path, 'r') as f:

        #print(beam_data)
        for i, beam_stack in enumerate(beam_data):
            print(beam_stack.shape)
            

(3, 288, 102)
(3, 288, 102)
(3, 288, 102)
(3, 288, 102)
(3, 288, 102)
(3, 288, 102)
(3, 288, 102)
(3, 288, 102)
(3, 288, 102)
(3, 288, 102)
(3, 288, 102)
(3, 288, 102)
(3, 288, 102)
(3, 288, 102)
(3, 288, 102)


In [6]:
#Example:
#if num_files is 20, then len(full_dataset) will be 60 (*3) because of the 3 beams

print(len(train_files))
print(len(val_files))
print(len(test_files))




2767
795
393


In [12]:
#DEBUG
import src.utils
importlib.reload(src.utils)

from src.utils import get_class_weights

In [13]:
# Cell 4: Initialize Model and Loss
num_classes = 6 # full_dataset.num_classes
model = TemporalCNN(input_channels=3, num_classes=num_classes)

#class_weights = torch.tensor([1/300, 1, 1, 1, 1, 1])                 # FIX THIS LATER -  TEMPORARY SETTING - DEFINE MANUALLY
class_weights = get_class_weights(train_dataset, num_classes)                    # FIX THIS LATER -  UNCOMMENT THIS OR DEFINE MANUALLY BUT CORRECT WEIGHTS
print(class_weights)
#tensor([1.8522e-01, 2.0864e+00, 2.3713e+01, 9.4286e+01, 2.3760e+04, 1.4505e+01]) # For 200 files, is inverse of [5.3990, 0.4793, 0.0422, 0.0106, 0.0000,0.0689]
# tensor([2.2450e-01, 3.8812e+01, 1.4015e+02, 9.8140e+02, 1.9692e+00, 9.9600e-01]) # For full dataset (or 70% anyways)
loss_fn = combined_loss(class_weights, alpha=config.loss_alpha)
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)


tensor([1.6761e-01, 3.8812e+01, 1.4015e+02, 9.8140e+02, 0.0000e+00, 0.0000e+00])


In [14]:
#DEBUG: reload train_model

import importlib
import src.utils
importlib.reload(src.utils)

from src.utils import seed_everything, get_class_weights, combined_loss, train_model

In [ ]:
# DEBUG:  One batch from your DataLoader

import os
os.environ["TORCH_DISABLE_MKL"] = "1"  # optional: disables MKL
os.environ["ONEDNN_VERBOSE"] = "0"
os.environ["DNNL_VERBOSE"] = "0"
torch.backends.mkldnn.enabled = False


x_batch, y_batch, meta_batch = next(iter(train_loader))

print(x_batch.shape)  # should be (B, T, num_classes)

print(type(x_batch))             # Should be torch.Tensor
print(x_batch.dtype)             # Should be torch.float32
print(torch.isnan(x_batch).any())  # Should be False
print(x_batch.shape)             # Should be [B, 3, T, R]

x_batch = x_batch.to(device)
model = model.to(device)

# Run forward pass
with torch.no_grad():
   outputs = model(x_batch) # (B, T, num_classes)

#optimizer.zero_grad()
#outputs = model(x_batch)     
print("output shape:", outputs.shape)
print("y shape:", y_batch.shape)

#Need to reshape the outputs, and y_batch for loss_fn to work properly:
outputs = outputs.reshape(-1, outputs.shape[-1])  # (B*T, num_classes)
y_batch = y_batch.view(-1)                        # (B*T, )

#Run the loss function
loss_fn = combined_loss(class_weights, alpha=config.loss_alpha)
loss = loss_fn(outputs, y_batch)

#Check the meta data contents 
print(meta_batch)

torch.Size([2, 3, 288, 102])
<class 'torch.Tensor'>
torch.float32
tensor(False)
torch.Size([2, 3, 288, 102])


In [19]:
print(loss)

tensor(1.4480)


In [15]:
# Cell 5: Train

#I was having bugs in this, where it was saying it failed creating a primitive. This was found to solve the issue (but shouldn't be used when proper training/running)

DEBUG_MODE = 0
if DEBUG_MODE:
    import os
    os.environ["TORCH_DISABLE_MKL"] = "1"  # optional: disables MKL
    os.environ["ONEDNN_VERBOSE"] = "0"
    os.environ["DNNL_VERBOSE"] = "0"
    torch.backends.mkldnn.enabled = False

if USE_WANDB:
    model = train_model(model, train_loader, val_loader, optimizer, loss_fn, device, num_epochs=config.epochs, patience=5, USE_WANDB=USE_WANDB)
else:
    model, history = train_model(model, train_loader, val_loader, optimizer, loss_fn, device, num_epochs=config.epochs, patience=5, USE_WANDB=USE_WANDB)

# ([2, 3, 288, 102])
# => [batch, channels, time, range]
    
#Best result using ce and dice-loss:
#Epoch 7/20 | Train Loss: 0.7003 | Val Loss: 0.9850 | Val F1: 0.4931
#⏹️ Early stopping triggered.
    
# With ce and graduated dice-loss:
# Starting Validation on Epoch #  6
# Epoch 7/20 | Train Loss: 0.7778 | Val Loss: 0.9043 | Val F1: 0.4931
# #
# Starting Validation on Epoch #  7
# Epoch 8/20 | Train Loss: 0.7598 | Val Loss: 0.9866 | Val F1: 0.4931
# ⏹️ Early stopping triggered.



Starting Training on Epoch #  0
Loss (& total) on Batch #1: 1.371364712715149 (1.371364712715149)
Loss (& total) on Batch #2: 1.3268176317214966 (2.6981823444366455)
Loss (& total) on Batch #3: 1.1413211822509766 (3.839503526687622)
Loss (& total) on Batch #4: 1.270466685295105 (5.109970211982727)
Loss (& total) on Batch #5: 1.254273772239685 (6.364243984222412)
Loss (& total) on Batch #6: 1.2321553230285645 (7.596399307250977)
Loss (& total) on Batch #7: 1.207209587097168 (8.803608894348145)
Loss (& total) on Batch #8: 1.1255338191986084 (9.929142713546753)
Loss (& total) on Batch #9: 1.1691280603408813 (11.098270773887634)
Loss (& total) on Batch #10: 1.1578707695007324 (12.256141543388367)
Loss (& total) on Batch #11: 1.1365044116973877 (13.392645955085754)
Loss (& total) on Batch #12: 1.1088361740112305 (14.501482129096985)
Loss (& total) on Batch #13: 1.0883320569992065 (15.589814186096191)
Loss (& total) on Batch #14: 1.6897883415222168 (17.279602527618408)
Loss (& total) on Batc

In [ ]:
#Profile memory from linux terminal with:
# $top -u slonimer

#8.2 g : Load data loaders
#1.4 g: More memory needed for running the model

#Notes: Running batch size of 16 for full data set takes a long time.  Should try doing larger batch size.
# No 2-power batch sizes are divisible by 3 (number of beams) but could try something larger, like 256


In [6]:
print(history)

{'train_loss': [0.9583881768968797, 0.8589790086111715, 0.8574826263131634, 0.8522834172171931, 0.8336483220900258, 0.8325665047572505, 0.82156113390961, 0.8176933988448112, 0.8018415827424296, 0.7963504906623594, 0.784831604890285, 0.7852150708917649, 0.7825517070389563, 0.7698381033635908, 0.7750539250912205, 0.7535960888189654, 0.751354037513656, 0.7524507615354753, 0.7423527644526574, 0.7395329321584394], 'val_loss': [0.8274070297678312, 0.792217128806644, 0.7939501139852736, 0.7924235314130783, 0.7700290646817949, 0.7498496580455039, 0.7518935592638122, 0.7520300754242473, 0.7267159074544907, 0.7344969949788518, 0.7293031530247794, 0.7422766718599532, 0.7212113978134261, 0.7469947520229552, 0.7144439145922661, 0.713933002617624, 0.7212710918651687, 0.7030177207456695, 0.7185809653666284, 0.7074187878105376], 'train_f1': [0.15546289138459493, 0.19007597932988193, 0.18941585088207896, 0.19484925846918627, 0.20334456890318764, 0.21256885120545368, 0.23521618564485366, 0.2263456753241

In [24]:
# Cell 6: Load Best Model and Evaluate
model.load_state_dict(torch.load("best_model_20250508.pt"))
#model.load_state_dict(torch.load("best_model.pt"))
model.eval()

all_preds = []
all_labels = []

#for x, y in val_loader:
for x, y, _ in test_loader:
    x = x.to(device)
    with torch.no_grad():
        out = model(x)
        #Need to reshape the outputs, and y to match dimensions:
        out = out.reshape(-1, out.shape[-1])  # (B*T, num_classes)
        #The prediction is the class with largest score per sample
        preds = torch.argmax(out, dim=1)

    #Need to reshape the outputs, and y to match dimensions:
    y = y.view(-1)                        # (B*T, )

    #Append the results
    all_preds.append(preds.cpu())
    all_labels.append(y)

y_pred = torch.cat(all_preds).numpy()
y_true = torch.cat(all_labels).numpy()

from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred))




/tmp/ipykernel_3219469/1187203441.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model_20250508.pt"))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       817
           1       0.98      0.98      0.98        47

    accuracy                           1.00       864
   macro avg       0.99      0.99      0.99       864
weighted avg       1.00      1.00      1.00       864



In [ ]:
'''

#Best model 2025-05-08

val_loader:
    
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    684575
           1       0.78      0.98      0.87      1913
           2       0.00      0.00      0.00       383
           3       0.44      0.89      0.59         9

    accuracy                           1.00    686880
   macro avg       0.56      0.72      0.62    686880
weighted avg       1.00      1.00      1.00    686880



test_loader:

           0       1.00      1.00      1.00    337877
           1       0.73      0.95      0.83      1257
           2       0.00      0.00      0.00       418
           3       0.00      0.00      0.00         0

    accuracy                           1.00    339552
   macro avg       0.43      0.49      0.46    339552
weighted avg       1.00      1.00      1.00    339552


'''

In [ ]:
#Cell 7: Make test plots.  
#I want to push a file through the algorithm, and see how it performs

#IMPORT EVERYTHING:
import os
import torch
from torch.utils.data import DataLoader, random_split

from dataset_loader_noEmbed import ADCPDataset  # your custom dataset
# from src.dataset_loader import ADCPDataset  # your custom dataset
from src.model import TemporalCNN # CNNClassifier  # your model
from src.utils import seed_everything, get_class_weights, combined_loss, train_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

'''
#DEBUG:
#I was having bugs in this, where it was saying it failed creating a primitive. This was found to solve the issue (but shouldn't be used when proper training/running)
import os
os.environ["TORCH_DISABLE_MKL"] = "1"  # optional: disables MKL
os.environ["ONEDNN_VERBOSE"] = "0"
os.environ["DNNL_VERBOSE"] = "0"
torch.backends.mkldnn.enabled = False
'''


# INITIALIZE THE MODEL
model_path = "/scratch/slonimer/ML_ADCP/"
num_classes = 6 
model = TemporalCNN(input_channels=3, num_classes=num_classes)
# model.load_state_dict(torch.load(model_path + "best_model_20250505.pt" ))
model.load_state_dict(torch.load(model_path + "best_model_20250508.pt" ))
model.eval()


#Specify the data to use:
file_path = "/scratch/slonimer/ML_ADCP/BACAX_24hr_h5/"
h5_filename = '20240406T000000_20240406T235959.h5'
#h5_filename = '20230701T000230_20230702T000229.h5'
h5_test_file = []
# h5_test_file.append(file_path + '20230701T000230_20230702T000229.h5') 
h5_test_file.append(file_path + h5_filename) 


#CLASSIFY THE TEST FILE
def classify_test_data(model, h5_test_file):
    test_file_dataset = ADCPDataset(h5_test_file)
    test_loader = DataLoader(test_file_dataset, batch_size=3, shuffle=False, num_workers=4)

    all_preds = []
    all_labels = []

    for x, y, meta in test_loader:
        x = x.to(device)
        model = model.to(device)  # ← Add this line
        with torch.no_grad():
            out = model(x)
            #Need to reshape the outputs, and y to match dimensions:
            out = out.reshape(-1, out.shape[-1])  # (B*T, num_classes)
            #The prediction is the class with largest score per sample
            preds = torch.argmax(out, dim=1)

        #Need to reshape the outputs, and y to match dimensions:
        y = y.view(-1)                        # (B*T, )

        #Append the results
        all_preds.append(preds.cpu())
        all_labels.append(y)
        
    return x, y, preds, meta


#Run the classification
x, y, preds, meta = classify_test_data(model, h5_test_file)

/tmp/ipykernel_1775809/1511681119.py:31: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path + "best_model_20250508.pt" ))


In [25]:
from sklearn.metrics import classification_report
print(classification_report(y.cpu(), preds.cpu()))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       817
           1       0.98      0.98      0.98        47

    accuracy                           1.00       864
   macro avg       0.99      0.99      0.99       864
weighted avg       1.00      1.00      1.00       864



In [ ]:
annotations[2]
#For '20230701T000230_20230702T000229.h5', the results are quite wrong
#Beam 1 and 3 include class 5, which is not good
#Beam 3 class 1 label start early, and ends too early, and goes to 0 for last 5 minutes

In [ ]:
preds.cpu()

In [49]:
import h5py
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np


def get_segments(annotations, ann):
    if ann==1: #For annotations (may be multiclass)
        mask = np.diff(annotations) != 0 # Create a mask of where changes in annotations are non-zero
        #diffs = mask.astype(int)
    elif ann==0: #For predictions
        mask = annotations != 0 # Create a mask of where annotations are non-zero
        
    diffs = np.diff(mask.astype(int)) # Find the changes in the mask
    start_indices = np.where(diffs == 1)[0] + 1 # Start indices: where diff == 1 (0 → 1)
    end_indices = np.where(diffs == -1)[0] + 1 # End indices: where diff == -1 (1 → 0)
    # Handle edge cases: 
    if mask[0]: #if it starts with a non-zero 
        start_indices = np.r_[0, start_indices]

    if mask[-1]: # if it ends with a non-zero
        end_indices = np.r_[end_indices, len(annotations)]

    anomaly_segments = list(zip(start_indices, end_indices)) # Zip together
    return anomaly_segments


def plot_results(x, annotations, predictions, filename, meta) :
    x = x.cpu()
    annotations = annotations.cpu()
    predictions = predictions.cpu()

    n_beams = x.shape[0]#[2]
    n_channels = x.shape[1]#[2]

    time_data = meta['time']

    for beam in range(n_beams):
        fig, axs = plt.subplots(n_channels, 1, sharex=True, figsize=(12, 2.5*n_channels))
        if n_channels == 1:
            axs = [axs]

        #Get the annotations for this beam
        anomaly_segments = get_segments(annotations[beam].cpu().numpy(),ann = 1)
        pred_segments = get_segments(predictions[beam].cpu().numpy(), ann = 0)

        #print(anomaly_segments)
        #print(pred_segments)

        #Determine if any annotations present in this beam:
        cls_str = '' #Initialize as nothing
        ann = annotations[beam].cpu().numpy()
        if np.any(ann>0):
            cls = np.median(ann[ann>0])
            cls_str = ', class: {}'.format(int(cls))


        #Plot Velocity, backscatter, and correlation, for each beam
        for ch in range(n_channels):
            #Plot the Complex Data
            im = axs[ch].imshow(
                x[beam,ch,:,:].T, aspect='auto', origin='lower',
                #extent=[extent[0], extent[1], extent[2], extent[3]],
                #extent=[t_hours[0], t_hours[-1], 0, arrp.shape[1]-1],
                interpolation='nearest',
                cmap='jet',
            )

            #Set the figure title
            if beam == 0 and ch == 0:
                fig.suptitle('File: {}'.format(filename))
            
            #Set the subplot titles
            if ch == 0:
                axs[ch].set_title('Beam #{} {}'.format(beam+1, cls_str))   
           
            #Add labels and titles
            #axs[ch].set_ylabel("Range bin" if range_dim is not None else '')
            #axs[ch].set_title(f"{var} - Channel {ch+1}")

            #Add dashed vertical lines for predictions
            for start, end in pred_segments:
                if 0 <= start < x.shape[2]:
                    axs[ch].axvline(x=start, color='black', linestyle='dashed', alpha=0.7)
                if 0 <= end < x.shape[2]:
                    axs[ch].axvline(x=end, color='black', linestyle='dashed', alpha=0.7)

            #Add shading for annotations
            for start, end in anomaly_segments:
                if 0 <= start < x.shape[2] and 0 <= end <= x.shape[2]:
                    axs[ch].axvspan(start, end, color='black', alpha=0.3)

            #Add a colorbar
            fig.colorbar(im, ax=axs[ch], label='color')

        # -- Date formatting for X --
        axs[-1].xaxis_date()  # tells matplotlib to interpret x as dates
        axs[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        fig.autofmt_xdate()  # Makes dates pretty (auto-rotates, etc.)

        #axs[-1].set_xlabel(time_dt[0].astype('datetime64[D]').astype(str))   # 'yyyy-mm-dd' date for xlabel
        #axs[-1].set_xlabel("Time (hours since start)")
        #fig.suptitle(f"{var} (shape={arr.shape})")
        plt.tight_layout()
        
        #if outdir:
        #    if not os.path.exists(outdir):
        #        os.makedirs(outdir)
        #    plt.savefig(f"{outdir}/{var}.png", dpi=120)
        #if show:
        #    plt.show()
        plt.show()
        #plt.close()

        
#

In [ ]:
#Get indices of start/end segments of anomalies
annotations = y.view(3,288)
predictions = preds.view(3,288)

plot_results(x, annotations, predictions, h5_filename, meta)

In [12]:
import h5py 
#path = file_path + '20230701T000230_20230702T000229.h5'

#WEIRD. THESE HAVE  THE SAME ANNOTATIONS AS ABOVE...
#path = file_path + '20240629T235731_20240630T235730.h5'
#path = file_path + '20240702T000000_20240702T235959.h5'
path = file_path + '20240406T000000_20240406T235959.h5'

with h5py.File(path, 'r') as f:
    if 'annotations' in f:
        print(annotations)
        #annotations = self._parse_annotations(f['annotations'])
    else:
        annotations = []
        

#CLASS 1: 'Dropout A',  '2024-07-01 22:30:00',	'2024-07-02 18:32:00',	'Beam 1'
#CLASS 5: 'MINOR B',	'2021-08-01 00:00:00',	'2023-11-01 00:00:00.0',	'Beam 1 and beam 3, MAYBE FINE. Lower intensity'



tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 

In [ ]:
#For ALL anomalous files, plot the annotations and the prediction

# Load anomaly filenames from a text file
with open("annotated_files.txt", "r") as f:
    anomaly_files = set(line.strip() for line in f if line.strip())
    
#Define anomaly paths before truncating h5_paths
file_path = "/scratch/slonimer/ML_ADCP/BACAX_24hr_h5/"
anomaly_paths = []
for filename in anomaly_files:
    anomaly_paths.append(file_path + filename)
    
#    anomaly_paths = [p for p in h5_paths if os.path.basename(p) in anomaly_files]

#Run the classification
for anomaly_path in anomaly_paths:
    print(anomaly_path)
    #Predict the class
    x, y, preds, meta = classify_test_data(model, [anomaly_path])
    
    #Get indices of start/end segments of anomalies
    annotations = y.view(3,288)
    predictions = preds.view(3,288)


    #Plot the results
    plot_results(x, annotations, predictions, os.path.basename(anomaly_path))

In [ ]:
#Look for False positives!

#Push all files through. If any are classified as drop-outs with more than 6 in a row (half an hour), make a plot

#Get a list of files from the directory:
data_folder = "/scratch/slonimer/ML_ADCP/BACAX_24hr_h5/"
#data_folder= r"F:\Documents\Projects\ML\ADCP_ML\h5_24h_files\\"
file_list = os.listdir(data_folder)
h5_files = {k for k in file_list if os.path.splitext(k)[1] == ".h5"}
#Files are inherently NOT in order in python! So if you want them in order, need to do this:
h5_files = sorted(h5_files)  # Sorts alphabetically

h5_paths = []
for filename in h5_files:
    h5_paths.append(data_folder + filename) 

'''
#For ALL anomalous files, plot the annotations and the prediction

# Load anomaly filenames from a text file
with open("annotated_files.txt", "r") as f:
    anomaly_files = set(line.strip() for line in f if line.strip())
    
#Define anomaly paths before truncating h5_paths
file_path = "/scratch/slonimer/ML_ADCP/BACAX_24hr_h5/"
anomaly_paths = []
for filename in anomaly_files:
    anomaly_paths.append(file_path + filename)
    
#    anomaly_paths = [p for p in h5_paths if os.path.basename(p) in anomaly_files]
'''

#Run the classification
for file_path in h5_paths:
    print(file_path)
    #Predict the class
    x, y, preds, meta = classify_test_data(model, [file_path])
    
    #Get indices of start/end segments of anomalies
    annotations = y.view(3,288)
    predictions = preds.view(3,288)

    
    #Determine if any annotations present in any beam:
    do_plot = 0
    for beam in range(3):
        ann_beam = annotations[beam].cpu().numpy()
        pred_beam = predictions[beam].cpu().numpy()
        
        #If more than one hour predicted in a day in any beam:
        n_samples = 24 # 1 hour
        #n_samples = 12 # 1 hour
        if np.all(ann_beam==0) and np.sum(pred_beam>0)>n_samples:
            do_plot = 1
    
    if do_plot == 1:
        #Plot the results
        plot_results(x, annotations, predictions, os.path.basename(file_path))

    